<a id='1'></a>
## 1 - Loading the libraries

In [90]:
import json
# import cohere
from google import genai
from groq import Groq
import os
from dotenv import load_dotenv
import time
from classifier import language_inference
import json
from functools import lru_cache
load_dotenv()

True

<a id='2'></a>
## 2 - Setting Configurations

In [91]:
# CONSTANTS
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
GROQ_API_KEY=os.getenv("GROQ_API_KEY")
MODEL_NAME = "aya-expanse-32b"
INTENTS = [
    "greeting",
    "goodbye",
    "gratitude",
    "asking_mental_health_question",
    "out_of_scope"
]

CONFIDENCE_THRESHOLD = 0.65

predictor = language_inference.LanguagePredictor()

In [92]:
# Client
gemini_client = genai.Client(api_key=GEMINI_API_KEY)
groq_client = Groq(api_key=GROQ_API_KEY)

In [93]:
# SYSTEM PROMPT
def build_intent_classifier_prompt(user_message: str) -> str:

    return f"""
You are an intent classification engine.

Your task is to classify the user's message into EXACTLY ONE intent.

Possible intents:

1. greeting
Definition:
The user is greeting, saying hello, starting a conversation,
or making a casual salutation.

2. goodbye
Definition:
The user is ending the conversation or saying farewell.

3. gratitude
Definition:
The user is thanking, appreciating,
or expressing gratitude.

4. asking_mental_health_question
Definition:
The user is discussing emotions, sadness, anxiety,
depression, stress, loneliness, emotional struggles,
mental health concerns, or asking psychological support questions.

Includes:
- depression
- anxiety
- panic
- loneliness
- stress
- emotional pain
- hopelessness
- emotional support seeking

5. out_of_scope
Definition:
The message does not belong to any previous category.

Rules:
- Always return ONLY valid JSON.
- Never explain.
- Never output markdown.
- Choose exactly one intent.
- Detect the intent regardless of language.
- Handle mixed-language text.
- Consider semantic meaning, not keywords only.

Allowed intents:
{INTENTS}

Examples:

User: "hello"
Output:
{{"intent":"greeting","confidence":0.99}}

User: "مرحبا كيف حالك"
Output:
{{"intent":"greeting","confidence":0.98}}

User: "hola amigo"
Output:
{{"intent":"greeting","confidence":0.98}}

User: "goodbye my friend"
Output:
{{"intent":"goodbye","confidence":0.99}}

User: "مع السلامة"
Output:
{{"intent":"goodbye","confidence":0.98}}

User: "thanks for helping me"
Output:
{{"intent":"gratitude","confidence":0.99}}

User: "شكرا جدا"
Output:
{{"intent":"gratitude","confidence":0.98}}

User: "I feel depressed and lonely"
Output:
{{"intent":"asking_mental_health_question","confidence":0.97}}

User: "انا حاسس باكتئاب"
Output:
{{"intent":"asking_mental_health_question","confidence":0.98}}

User: "انا تعبان نفسيا today"
Output:
{{"intent":"asking_mental_health_question","confidence":0.97}}

User: "what is the weather today"
Output:
{{"intent":"out_of_scope","confidence":0.99}}

Now classify this message:

User: "{user_message}"
"""

In [103]:
# LLM Call
def call_llm(prompt: str, llm_model="Gemini") -> dict:
    if llm_model=="Gemini":
        response = gemini_client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt,
            config={
                "temperature": 0,
                "response_mime_type": "application/json"
            }
        )

        return json.loads(response.text)

    
    elif llm_model=="Groq":
        response = groq_client.chat.completions.create(
            model="llama-3.1-8b-instant",
            temperature=0,
            response_format={"type": "json_object"},
            messages=[
                {
                    "role": "system",
                    "content": prompt
                }
            ]
        )

        content = response.choices[0].message.content

        return json.loads(content)

In [95]:
def validate_intent_response(response: dict) -> dict:

    if "intent" not in response:
        return {
            "intent": "out_of_scope",
            "confidence": 0.0
        }

    if "confidence" not in response:
        response["confidence"] = 0.0

    if response["intent"] not in INTENTS:
        return {
            "intent": "out_of_scope",
            "confidence": 0.0
        }

    return response

In [96]:
# INTENT CLASSIFICATION
def classify_intent(user_message: str, llm_model="Gemini") -> dict:

    prompt = build_intent_classifier_prompt(user_message)

    llm_response = call_llm(prompt, llm_model)

    validated_response = validate_intent_response(llm_response)

    language = predictor.predict(user_message)

    validated_response["language"] = language

    return validated_response

In [97]:
def apply_confidence_threshold(intent_data: dict) -> dict:

    confidence = intent_data["confidence"]

    if confidence < CONFIDENCE_THRESHOLD:
        intent_data["intent"] = "out_of_scope"
        intent_data["confidence"] = confidence

    return intent_data

In [98]:
# Translate
@lru_cache(maxsize=None)
def load_locale(language: str):
    with open(f"locales/{language}.json", "r", encoding="utf-8") as f:
        return json.load(f)

# Static Response
def build_response(intent: str, language: str):

    locales = load_locale(language)

    # fallback to English if language missing
    if intent not in locales:
        english_locales = load_locale("en")
        return english_locales.get(intent, "")

    return locales[intent]

In [99]:
# Hanlder
def greeting_handler(user_message: str, language: str):
    return build_response("greeting", language)


def goodbye_handler(user_message: str, language: str):
    return build_response("goodbye", language)


def gratitude_handler(user_message: str, language: str):
    return build_response("gratitude", language)


def mental_health_handler(user_message: str, language: str):
    return build_response("asking_mental_health_question", language)


def out_of_scope_handler(user_message: str, language: str):
    return build_response("out_of_scope", language)

In [100]:
# =========================================================
# MAIN ROUTER
# =========================================================
def route_intent(user_message: str, intent_data: dict):

    intent_data = apply_confidence_threshold(intent_data)

    intent = intent_data["intent"]
    language = intent_data["language"]

    if intent == "greeting":
        return greeting_handler(user_message, language)

    elif intent == "goodbye":
        return goodbye_handler(user_message, language)

    elif intent == "gratitude":
        return gratitude_handler(user_message, language)

    elif intent == "asking_mental_health_question":
        return mental_health_handler(user_message, language)

    else:
        return out_of_scope_handler(user_message, language)

In [101]:
# FULL PIPELINE
# =========================================================
def chatbot(user_message: str, intent_data: dict):

    response = route_intent(
        user_message,
        intent_data
    )

    return response

In [102]:
examples = [
    "مرحبا",
    "شكرا جدا",
    "I feel anxious and lonely",
    "انا تعبان نفسيا",
    "what is the weather today",
    "bye"
]

for msg in examples:

    print("=" * 50)

    print("USER:", msg)

    intent_data = classify_intent(
        msg,
        llm_model="Groq"
    )

    print("INTENT:", intent_data)

    final_response = chatbot(
        msg,
        intent_data
    )

    print("BOT:", final_response)

    time.sleep(15)

USER: مرحبا
Groq
INTENT: {'intent': 'greeting', 'confidence': 0.98, 'language': np.str_('ar')}
BOT: مرحبًا 👋 كيف يمكنني مساعدتك اليوم؟
USER: شكرا جدا
Groq
INTENT: {'intent': 'gratitude', 'confidence': 0.98, 'language': np.str_('ar')}
BOT: على الرحب والسعة، سعيد بمساعدتك.
USER: I feel anxious and lonely
Groq
INTENT: {'intent': 'asking_mental_health_question', 'confidence': 0.98, 'language': np.str_('en')}
BOT: I'm here to listen. Would you like to talk more about what you're feeling?
USER: انا تعبان نفسيا
Groq
INTENT: {'intent': 'asking_mental_health_question', 'confidence': 0.98, 'language': np.str_('ar')}
BOT: أنا هنا للاستماع. هل ترغب في التحدث أكثر عما تشعر به؟
USER: what is the weather today
Groq
INTENT: {'intent': 'out_of_scope', 'confidence': 0.99, 'language': np.str_('en')}
BOT: I'm designed mainly for mental health support, greetings, and basic conversational interactions.
USER: bye
Groq
INTENT: {'intent': 'goodbye', 'confidence': 0.99, 'language': np.str_('sw')}
BOT: Jitunze. 